# Day 11 — Agent Memory, Deployment & Reliability (hands-on)

Companion notebook to [`../notes.md`](../notes.md). Seven small steps, all using LangGraph, all
**without an API key** (the "chatbot" is a plain function, so we can focus on how memory and reliability
work — a real model would read exactly the same `messages` list).

| Step | Question it answers |
|---|---|
| 1 | How does a chatbot remember things *inside* one conversation? |
| 2 | How does it remember things *across* conversations? |
| 3 | How does memory survive a restart? |
| 4 | How do I put the graph behind an API? |
| 5 | What if a step fails? (retry) |
| 6 | What if a graph loops forever? (safety limit) |
| 7 | What if it crashes halfway? (resume) |

In [1]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

## 1. Short-term memory — one conversation

Short-term memory is just the graph's **state**, saved by a **checkpointer** (Day 10) under a
`thread_id`. Same `thread_id` = same conversation = it remembers. New `thread_id` = a fresh start.

Our chatbot node simply lists what the user has said so far, so we can *see* the memory working.

In [2]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: ChatState) -> dict:
    said = [m.content for m in state["messages"] if m.type == "human"]
    reply = f"You have said {len(said)} thing(s) so far: {said}"
    return {"messages": [AIMessage(content=reply)]}


builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile(checkpointer=MemorySaver())

In [3]:
alice = {"configurable": {"thread_id": "alice-chat"}}
bob = {"configurable": {"thread_id": "bob-chat"}}

first = graph.invoke({"messages": [HumanMessage("hi, I'm Alice")]}, alice)
second = graph.invoke({"messages": [HumanMessage("I like tea")]}, alice)
other = graph.invoke({"messages": [HumanMessage("hello")]}, bob)

print("Alice, message 1:", first["messages"][-1].content)
print("Alice, message 2:", second["messages"][-1].content)
print("Bob,   message 1:", other["messages"][-1].content)

Alice, message 1: You have said 1 thing(s) so far: ["hi, I'm Alice"]
Alice, message 2: You have said 2 thing(s) so far: ["hi, I'm Alice", 'I like tea']
Bob,   message 1: You have said 1 thing(s) so far: ['hello']


Alice's second message knew about her first — same thread. Bob's thread started fresh.

**The catch:** this memory belongs to *one thread*. Open a new chat and it's gone.

## 2. Long-term memory — a Store shared across conversations

Facts like "Raj prefers short answers in Telugu" should follow Raj into **every** new chat. For that,
LangGraph has a **Store**: a key-value box, organised by **namespace** (here: `("users", "raj")`).

First, the store on its own:

In [4]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

store.put(("users", "raj"), "preferences", {"language": "Telugu", "style": "short answers"})

item = store.get(("users", "raj"), "preferences")
print(item.value)

{'language': 'Telugu', 'style': 'short answers'}


Now a graph node that reads from the store. When you compile a graph with `store=...`, LangGraph hands
the store to any node that asks for it. The user's id arrives through `config`.

In [5]:
from langgraph.store.base import BaseStore


def greet(state: ChatState, config, *, store: BaseStore) -> dict:
    user_id = config["configurable"]["user_id"]
    item = store.get(("users", user_id), "preferences")
    prefs = item.value if item else "nothing saved yet"
    return {"messages": [AIMessage(content=f"Hello {user_id}! What I remember about you: {prefs}")]}


memory_builder = StateGraph(ChatState)
memory_builder.add_node("greet", greet)
memory_builder.add_edge(START, "greet")
memory_builder.add_edge("greet", END)

memory_graph = memory_builder.compile(checkpointer=MemorySaver(), store=store)

In [6]:
for thread_id, user_id in [("monday-chat", "raj"), ("tuesday-chat", "raj"), ("first-chat", "sam")]:
    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}}
    result = memory_graph.invoke({"messages": [HumanMessage("hi")]}, config)
    print(f"{thread_id:>13}: {result['messages'][-1].content}")

  monday-chat: Hello raj! What I remember about you: {'language': 'Telugu', 'style': 'short answers'}
 tuesday-chat: Hello raj! What I remember about you: {'language': 'Telugu', 'style': 'short answers'}
   first-chat: Hello sam! What I remember about you: nothing saved yet


Two **different threads** (Monday and Tuesday), and both remember Raj — because the memory lives in
the **store**, not in the thread. Sam has nothing saved yet.

Short-term memory = **checkpointer** (per thread). Long-term memory = **store** (across threads).

## 3. Memory that survives a restart

`MemorySaver` and `InMemoryStore` live in RAM — restart the program and everything is gone. For real
apps, save to a database. Here we use a **SQLite file** (production systems use Postgres, same idea).

We'll simulate a restart: close the connection, throw the graph away, build a new one on the same file.

In [7]:
import sqlite3
import tempfile
import uuid
from pathlib import Path

from langgraph.checkpoint.sqlite import SqliteSaver

db_file = Path(tempfile.gettempdir()) / f"day11_memory_{uuid.uuid4().hex[:6]}.sqlite"
config = {"configurable": {"thread_id": "persistent-chat"}}

connection = sqlite3.connect(db_file, check_same_thread=False)
saved_graph = builder.compile(checkpointer=SqliteSaver(connection))
saved_graph.invoke({"messages": [HumanMessage("my favourite colour is blue")]}, config)

connection.close()
print("Program 'restarted': connection closed, graph object thrown away.")

Program 'restarted': connection closed, graph object thrown away.


In [8]:
connection = sqlite3.connect(db_file, check_same_thread=False)
new_graph = builder.compile(checkpointer=SqliteSaver(connection))

result = new_graph.invoke({"messages": [HumanMessage("do you remember me?")]}, config)
print(result["messages"][-1].content)
connection.close()

You have said 2 thing(s) so far: ['my favourite colour is blue', 'do you remember me?']


A brand-new graph on the same file still knew the earlier message. That's what a **persistent
checkpointer** gives you.

## 4. Deploying the graph behind an API

A server can't run inside notebook cells, so the API lives in
[`graph_api_example.py`](graph_api_example.py) — a small FastAPI app (like Day 08) with one
endpoint, `POST /chat`. The important part: **each request carries a `thread_id`**, and the app passes it
to the graph as the conversation id. The API is stateless; the *memory* lives in the SQLite file.

We can test it here without starting a server, using FastAPI's `TestClient`:

In [9]:
from fastapi.testclient import TestClient
from graph_api_example import create_app

api_db = Path(tempfile.gettempdir()) / f"day11_api_{uuid.uuid4().hex[:6]}.sqlite"
client = TestClient(create_app(str(api_db)))

requests = [("raj-1", "hi, I like tea"), ("raj-1", "what did I say?"), ("sam-1", "hello")]

for thread_id, text in requests:
    reply = client.post("/chat", json={"thread_id": thread_id, "message": text}).json()["reply"]
    print(f"{thread_id}: {reply}")

raj-1: You have said 1 thing(s) so far: ['hi, I like tea']


raj-1: You have said 2 thing(s) so far: ['hi, I like tea', 'what did I say?']


sam-1: You have said 1 thing(s) so far: ['hello']


Same `thread_id` → remembers. Different `thread_id` → separate conversation.

To run it for real: `uvicorn graph_api_example:create_app --factory --reload`.

LangGraph also has an official route: a `langgraph.json` file plus the `langgraph dev` command, which
serves your graph with a ready-made API (see `notes.md`, Section 2). We don't run that here.

## 5. Reliability: retry a step that fails

Day 08's retry idea, built into the graph: give a node a **`RetryPolicy`**. To test it, here is a step
that fails twice ("simulated timeout") and works on the third try.

In [10]:
from langgraph.types import RetryPolicy

calls = {"count": 0}


def flaky_step(state: ChatState) -> dict:
    calls["count"] += 1
    if calls["count"] < 3:
        raise TimeoutError("simulated timeout")
    return {"messages": [AIMessage(content=f"Worked on attempt {calls['count']}")]}


def make_graph(retry_policy=None):
    b = StateGraph(ChatState)
    b.add_node("flaky_step", flaky_step, retry_policy=retry_policy)
    b.add_edge(START, "flaky_step")
    b.add_edge("flaky_step", END)
    return b.compile()

In [11]:
calls["count"] = 0
try:
    make_graph().invoke({"messages": [HumanMessage("go")]})
except TimeoutError:
    print("Without a retry policy: the first failure crashed the whole run.")

Without a retry policy: the first failure crashed the whole run.


In [12]:
calls["count"] = 0
policy = RetryPolicy(max_attempts=4, initial_interval=0.1, retry_on=TimeoutError)

result = make_graph(policy).invoke({"messages": [HumanMessage("go")]})
print("With a retry policy:", result["messages"][-1].content)

With a retry policy: Worked on attempt 3


## 6. Reliability: never loop forever

A graph with a loop can get stuck (Day 09/10). LangGraph counts steps and stops with an error at the
**recursion limit**. Here is a deliberately broken graph whose node points back at itself:

In [13]:
from langgraph.errors import GraphRecursionError


def spin(state: ChatState) -> dict:
    return {"messages": [AIMessage(content="still going...")]}


loop_builder = StateGraph(ChatState)
loop_builder.add_node("spin", spin)
loop_builder.add_edge(START, "spin")
loop_builder.add_edge("spin", "spin")  # never reaches END!
loop_graph = loop_builder.compile()

try:
    loop_graph.invoke({"messages": [HumanMessage("go")]}, {"recursion_limit": 5})
except GraphRecursionError:
    print("Stopped by the safety limit after 5 steps, instead of running forever.")

Stopped by the safety limit after 5 steps, instead of running forever.


Always set a sensible limit on graphs that loop — an agent stuck in a loop burns time **and money**.

## 7. Reliability: resume after a crash (checkpoints, again)

A three-step job: **fetch → process → save**. The `process` step crashes because a service is down.
Thanks to the checkpointer (Day 10), the graph remembers `fetch` already finished — so after we fix the
problem, it resumes at `process`, **without redoing `fetch`**.

In [14]:
class JobState(TypedDict):
    log: list


service = {"is_down": True}
runs = {"fetch": 0, "process": 0, "save": 0}


def fetch(state: JobState) -> dict:
    runs["fetch"] += 1
    return {"log": state["log"] + ["fetched data"]}


def process(state: JobState) -> dict:
    runs["process"] += 1
    if service["is_down"]:
        raise ConnectionError("the processing service is down")
    return {"log": state["log"] + ["processed data"]}


def save(state: JobState) -> dict:
    runs["save"] += 1
    return {"log": state["log"] + ["saved result"]}


job_builder = StateGraph(JobState)
job_builder.add_node("fetch", fetch)
job_builder.add_node("process", process)
job_builder.add_node("save", save)
job_builder.add_edge(START, "fetch")
job_builder.add_edge("fetch", "process")
job_builder.add_edge("process", "save")
job_builder.add_edge("save", END)

job_graph = job_builder.compile(checkpointer=MemorySaver())
job_config = {"configurable": {"thread_id": "job-1"}}

In [15]:
try:
    job_graph.invoke({"log": []}, job_config)
except ConnectionError as error:
    print("CRASHED:", error)

snapshot = job_graph.get_state(job_config)
print("Progress saved so far:", snapshot.values["log"])
print("Waiting to run next  :", snapshot.next)

CRASHED: the processing service is down
Progress saved so far: ['fetched data']
Waiting to run next  : ('process',)


The graph is paused at `process`, with `fetch`'s result safely saved. The service comes back — resume by
calling `invoke(None, ...)` (no new input, just "continue"):

In [16]:
service["is_down"] = False

final = job_graph.invoke(None, job_config)

print("Final log:", final["log"])
print("Times each step ran:", runs)

Final log: ['fetched data', 'processed data', 'saved result']
Times each step ran: {'fetch': 1, 'process': 2, 'save': 1}


`fetch` ran **once**, `process` ran twice (the crash + the retry), `save` once. In a real system,
`fetch` might be an expensive API call or a paid model call — not redoing it is exactly the point.

## What we covered

| Need | Tool | Where it lives |
|---|---|---|
| Remember inside one conversation | Checkpointer + `thread_id` | Per thread |
| Remember across conversations | Store (namespace + key) | Shared across threads |
| Survive a restart | Persistent checkpointer (SQLite / Postgres) | A database file/server |
| Serve it to other apps | FastAPI (or `langgraph dev`) with `thread_id` per request | Stateless API |
| A step fails | `RetryPolicy` on the node | Node setting |
| A loop runs away | `recursion_limit` | Run config |
| Crash halfway | Checkpoint + `invoke(None, ...)` | Saved state |

## Try it yourself

- In step 2, `store.put` a preference for `"sam"` and re-run the loop — Sam now gets remembered too.
- In step 5, change `max_attempts=4` to `2` and watch the retry policy give up.
- In step 6, raise the `recursion_limit` to `50` — it still stops, just later.
- In step 7, leave `service["is_down"]` as `True` and call `invoke(None, ...)` again — it fails again
  at the same step, and `fetch` still doesn't re-run.